# LUẬN VĂN MULTIMODAL EXPLAINABLE AI: PHÂN LỚP 6 THỂ LOẠI DI SẢN VĂN HÓA PHI VẬT THỂ
## Phase 7: Chuẩn Bị Dữ Liệu Hình Ảnh & Pipeline Gán Nhãn Bán Tự Động (Semi-Automatic Concept Labeling)
**Tác giả:** Đồ án Tốt nghiệp Đại học - Nhóm Multimodal XAI Heritage  
**Phạm vi 6 thể loại:** Cải lương (`cai_luong`), Ca trù (`ca_tru`), Chèo (`cheo`), Chầu văn (`chau_van`), Quan họ (`quan_ho`), Hát xẩm (`hat_xam`)

---

### MỤC TIÊU VÀ BÀI TOÁN THỰC TẾ
Notebook này được thiết kế tối ưu để chạy trên **Google Colab (GPU T4 miễn phí)** hoặc **máy Local**, giải quyết triệt để 3 thách thức dữ liệu thị giác:
1. **Dữ liệu Video lệch thời lượng cực lớn:** Xử lý tình trạng lớp có 30 video ngắn 1-2 phút, lớp có 2-3 video dài 20 phút bằng cơ chế **Virtual Sub-clips 60s** kèm **Buffer Gap 15s bị loại bỏ** để chống rò rỉ bối cảnh (Data Leakage).
2. **Lọc trùng lặp cảnh và frame mờ:** Áp dụng phương pháp Perceptual Hash / Color Histogram Cosine Similarity kết hợp Laplacian Blur Variance để loại bỏ frame tĩnh khi máy quay đứng yên ở video dài.
3. **Khai thác tập Roboflow 107 classes:** Tự động lọc 21 categories đạo cụ/trang phục hữu ích, ánh xạ sang **25 Visual Concept Attributes nhị phân (0/1)** và xuất nhãn chuẩn YOLOv8 để huấn luyện trợ lý phát hiện đạo cụ.
4. **Pipeline Gán nhãn Bán tự động 2 Tầng:** 
   - *Tầng 1 (Sub-clip Level):* Xuất file sheet ~120 dòng kèm thumbnail để người dùng duyệt/sửa bằng mắt trong 20-30 phút.
   - *Tầng 2 (Frame-level Refinement):* Dùng Zero-shot CLIP (`openai/clip-vit-base-patch32`) và Roboflow Detector tinh chỉnh nhãn chính xác cho từng frame riêng lẻ.

### BƯỚC 0: Cài Đặt Môi Trường & Kiểm Tra Phần Cứng
Hỗ trợ cả môi trường **Google Colab** và **Local**. Tự động cài đặt các thư viện cần thiết (`opencv-python`, `transformers`, `torch`, `ultralytics`, `imagehash`).

In [ ]:
# 0. THIẾT LẬP MÔI TRƯỜNG, AUTO-MOUNT GOOGLE DRIVE & KIỂM TRA PHẦN CỨNG
import os
import sys
import json
import time
import math
import random
import shutil
import zipfile
from pathlib import Path

# Kiểm tra môi trường Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('[Google Colab] Đang khởi tạo môi trường Colab...')
    # Cài đặt các gói phụ thuộc nếu thiếu
    get_ipython().system('pip install -q opencv-python Pillow transformers torch torchvision scikit-learn tqdm imagehash ultralytics timm')

    # 1. Tự động kết nối Google Drive
    from google.colab import drive
    drive_mount_root = Path('/content/drive')
    if not (drive_mount_root / 'MyDrive').exists() and not (drive_mount_root / 'My Drive').exists():
        print('[Google Colab] Đang kết nối Google Drive...')
        drive.mount('/content/drive')
    else:
        print('[Google Colab] Google Drive đã được kết nối sẵn!')

    # 2. Tự động định vị thư mục dự án trên Google Drive
    possible_roots = [
        Path('/content/drive/MyDrive/Colab Notebooks/LuanVan'),
        Path('/content/drive/My Drive/Colab Notebooks/LuanVan'),
        Path('/content/drive/MyDrive/LuanVan'),
        Path('/content/drive/My Drive/LuanVan'),
        Path('/content/drive/MyDrive'),
        Path('/content')
    ]

    ROOT_DIR = None
    for p in possible_roots:
        if p.exists() and (
            (p / 'dataset').exists() or 
            (p / 'data').exists() or 
            (p / 'Visual_Data_Prep_and_SemiAuto_Labeling_Colab.ipynb').exists()
        ):
            ROOT_DIR = p
            break

    if ROOT_DIR is None:
        ROOT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LuanVan')
        ROOT_DIR.mkdir(parents=True, exist_ok=True)

    # Chuyển thư mục làm việc về ROOT_DIR để mọi đường dẫn tương đối hoạt động đồng nhất
    os.chdir(str(ROOT_DIR))
    print(f'[Google Colab] Đã chuyển thư mục làm việc về: {os.getcwd()}')
else:
    print('[Local Environment] Đang chạy trên máy cục bộ.')
    ROOT_DIR = Path('.')

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Kiểm tra GPU PyTorch
import torch
print('Python Version:', sys.version.split()[0])
print('PyTorch Version:', torch.__version__)
print('CUDA GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')

# Thiết lập Random Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Môi trường sẵn sàng làm việc!')


### BƯỚC 1: Cấu Hình Đường Dẫn & Khởi Tạo Danh Mục Visual Concepts Chuẩn
Đọc danh mục concept hình ảnh từ `concept/visual_concepts.md` và chuẩn hóa thành **25 thuộc tính nhị phân (0/1)** cốt lõi phân biệt 6 thể loại thuộc 3 nhóm:
- **Nhóm 1 (Nhạc cụ & Đạo cụ):** `guitar_phim_lom`, `dan_kim_dan_nguyet`, `dan_co_dan_nhi`, `dan_tranh`, `dan_bau`, `dan_day`, `ban_phach_trong_chau`, `trong_cheo_mo_quat_giay`, `dao_cu_hau_dong_kiem_cung_mam`, `senh_tien_trong_manh`, `khong_nhac_cu`.
- **Nhóm 2 (Trang phục & Hóa trang):** `co_trang_mao_giap_dinh_hat`, `ao_dai_ao_ba_ba`, `ao_thao_khan_nhieu_ao_the`, `ao_tu_than_yem_dao`, `trang_phuc_he_cheo`, `khan_chau_ao_ngu_hau_dong`, `non_quai_thao_khan_mo_qua_ao_mo_ba`, `ao_nau_kinh_den_mu_cat`.
- **Nhóm 3 (Không gian & Đội hình):** `san_khau_kich_hat`, `doi_hinh_3_nguoi_ca_tru`, `chieu_cheo_san_dinh`, `dien_tho_hau_dong`, `doi_dap_nam_nu_quan_ho`, `chieu_xam_duong_pho_cho`.

In [ ]:
# 1. CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU & DANH MỤC 25 VISUAL CONCEPTS
DATA_DIR = ROOT_DIR / 'data'
FRAMES_DIR = DATA_DIR / 'frames'
ANNOTATIONS_DIR = DATA_DIR / 'annotations'
YOLO_DIR = DATA_DIR / 'yolo_dataset'

# Tự động tạo các thư mục lưu trữ nếu chưa có
for d in [DATA_DIR, FRAMES_DIR, ANNOTATIONS_DIR, YOLO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Tự động dò tìm chính xác vị trí thư mục Video
# Hỗ trợ cả trường hợp nằm trong dataset/video/ hoặc nằm trực tiếp trong dataset/
possible_video_dirs = [
    ROOT_DIR / 'dataset' / 'video',
    ROOT_DIR / 'dataset',
    Path('/content/dataset/video'),
    Path('/content/dataset'),
    Path('/content/drive/MyDrive/Colab Notebooks/LuanVan/dataset/video'),
    Path('/content/drive/MyDrive/Colab Notebooks/LuanVan/dataset'),
    Path('/content/drive/My Drive/Colab Notebooks/LuanVan/dataset/video'),
    Path('/content/drive/My Drive/Colab Notebooks/LuanVan/dataset')
]

VIDEO_DATASET_DIR = None
for p in possible_video_dirs:
    if p.exists() and p.is_dir():
        sub_items = [item.name.lower() for item in p.iterdir() if item.is_dir()]
        if any(k in sub_items for k in ['cailuong', 'catru', 'cheo', 'chauvan', 'quanho', 'xam', 'video']):
            if 'video' in sub_items and (p / 'video').is_dir():
                VIDEO_DATASET_DIR = p / 'video'
            else:
                VIDEO_DATASET_DIR = p
            break

if VIDEO_DATASET_DIR is None:
    VIDEO_DATASET_DIR = ROOT_DIR / 'dataset' / 'video'

print(f'[Đường dẫn] Thư mục gốc (ROOT_DIR): {ROOT_DIR.resolve()}')
print(f'[Đường dẫn] Thư mục Video (VIDEO_DATASET_DIR): {VIDEO_DATASET_DIR.resolve()} (Tồn tại: {VIDEO_DATASET_DIR.exists()})')
if VIDEO_DATASET_DIR.exists():
    found_subdirs = [d.name for d in VIDEO_DATASET_DIR.iterdir() if d.is_dir()]
    print(f'  -> Các thư mục con tìm thấy: {found_subdirs}')

# Tự động dò tìm và giải nén roboflowdataset.zip nếu cần
ROBOFLOW_DIR = ROOT_DIR / 'roboflowdataset'
possible_zip = [
    ROOT_DIR / 'roboflowdataset.zip',
    Path('/content/roboflowdataset.zip'),
    Path('/content/drive/MyDrive/Colab Notebooks/LuanVan/roboflowdataset.zip'),
    Path('/content/drive/My Drive/Colab Notebooks/LuanVan/roboflowdataset.zip')
]

if not ROBOFLOW_DIR.exists() or len(list(ROBOFLOW_DIR.glob('*'))) == 0:
    for z in possible_zip:
        if z.exists():
            print(f'[Roboflow] Phát hiện {z.name}, đang tự động giải nén sang {ROBOFLOW_DIR}...')
            ROBOFLOW_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(str(z), 'r') as zip_ref:
                zip_ref.extractall(str(ROBOFLOW_DIR))
            print(f'[Roboflow] Giải nén thành công! Thư mục: {ROBOFLOW_DIR}')
            break

print(f'[Đường dẫn] Thư mục Roboflow: {ROBOFLOW_DIR.resolve()} (Tồn tại: {ROBOFLOW_DIR.exists()})')

# Bản đồ mã thể loại và tên hiển thị (hỗ trợ cả có dấu gạch dưới và viết liền)
GENRE_MAP = {
    'cailuong': 'Cải lương',
    'cai_luong': 'Cải lương',
    'catru': 'Ca trù',
    'ca_tru': 'Ca trù',
    'cheo': 'Chèo',
    'chauvan': 'Chầu văn',
    'chau_van': 'Chầu văn',
    'quanho': 'Quan họ',
    'quan_ho': 'Quan họ',
    'xam': 'Hát xẩm',
    'hat_xam': 'Hát xẩm',
    'hatxam': 'Hát xẩm'
}

# 25 Visual Concept Attributes nhị phân (0/1) chuẩn hóa
VISUAL_CONCEPT_ATTRIBUTES = {
    # Nhóm 1: Nhạc cụ & Đạo cụ (11 attributes)
    'guitar_phim_lom': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Đàn guitar phím lõm tạo âm sắc đặc trưng trong dàn nhạc Cải lương',
        'default_genres': ['Cải lương']
    },
    'dan_kim_dan_nguyet': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Đàn kìm (Cải lương) hoặc đàn nguyệt (Chầu văn)',
        'default_genres': ['Cải lương', 'Chầu văn']
    },
    'dan_co_dan_nhi': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Đàn cò / đàn nhị dẫn dắt giai điệu',
        'default_genres': ['Cải lương', 'Chầu văn', 'Hát xẩm']
    },
    'dan_tranh': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Đàn tranh (đàn thập lục) trong dàn nhạc tài tử hoặc Chầu văn',
        'default_genres': ['Cải lương', 'Chầu văn']
    },
    'dan_bau': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Đàn bầu (độc huyền cầm) với âm sắc trong trẻo trong Chèo, Hát Xẩm',
        'default_genres': ['Chèo', 'Hát xẩm']
    },
    'dan_day': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Đàn đáy cần đàn dài, thùng gỗ mộc âm trầm đặc trưng của Ca trù',
        'default_genres': ['Ca trù']
    },
    'ban_phach_trong_chau': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Bàn phách do đào nương tự gõ giữ nhịp và trống chầu do quan viên gõ điểm xao',
        'default_genres': ['Ca trù']
    },
    'trong_cheo_mo_quat_giay': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Trống Chèo, mõ, quạt giấy giữ nhịp và tương tác động tác múa',
        'default_genres': ['Chèo']
    },
    'dao_cu_hau_dong_kiem_cung_mam': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Đạo cụ nghi lễ hầu đồng: kiếm, cung, mâm vàng, lẵng hoa, nón quai thao giá Cô Bơ',
        'default_genres': ['Chầu văn']
    },
    'senh_tien_trong_manh': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Sênh tiền, sênh sứa, phách, trống mảnh giữ nhịp mộc mạc của Hát Xẩm',
        'default_genres': ['Hát xẩm']
    },
    'khong_nhac_cu': {
        'group': 'nhac_cu_dao_cu',
        'desc': 'Trình diễn thuần túy bằng giọng hát đối đáp giao duyên, không có nhạc cụ đệm',
        'default_genres': ['Quan họ']
    },

    # Nhóm 2: Trang phục & Hóa trang (8 attributes)
    'co_trang_mao_giap_dinh_hat': {
        'group': 'trang_phuc_hoa_trang',
        'desc': 'Phục trang cổ trang lụa gấm nhung, mũ mão, giáp đính hạt, họa tiết rồng phượng',
        'default_genres': ['Cải lương']
    },
    'ao_dai_ao_ba_ba': {
        'group': 'trang_phuc_hoa_trang',
        'desc': 'Áo dài truyền thống, áo bà ba trong các vở Cải lương đề tài xã hội hoặc nghệ sĩ',
        'default_genres': ['Cải lương', 'Quan họ']
    },
    'ao_thao_khan_nhieu_ao_the': {
        'group': 'trang_phuc_hoa_trang',
        'desc': 'Đào nương mặc áo thao quấn khăn nhiễu, kép đàn mặc áo the khăn xếp',
        'default_genres': ['Ca trù']
    },
    'ao_tu_than_yem_dao': {
        'group': 'trang_phuc_hoa_trang',
        'desc': 'Trang phục nữ dân gian Bắc Bộ: áo tứ thân, yếm đào, váy, vấn khăn',
        'default_genres': ['Chèo', 'Quan họ']
    },
    'trang_phuc_he_cheo': {
        'group': 'trang_phuc_hoa_trang',
        'desc': 'Trang phục hề Chèo phóng đại, sặc sỡ hỗ trợ tính trào lộng, hài hước',
        'default_genres': ['Chèo']
    },
    'khan_chau_ao_ngu_hau_dong': {
        'group': 'trang_phuc_hoa_trang',
        'desc': 'Trang phục hầu đồng rực rỡ (áo mớ ba, khăn ba múi, hài phượng, áo ngự các màu)',
        'default_genres': ['Chầu văn']
    },
    'non_quai_thao_khan_mo_qua_ao_mo_ba': {
        'group': 'trang_phuc_hoa_trang',
        'desc': 'Liền chị đầu vấn khăn mỏ quạ, đội nón quai thao, áo mớ ba mớ bảy, đeo xà tích',
        'default_genres': ['Quan họ']
    },
    'ao_nau_kinh_den_mu_cat': {
        'group': 'trang_phuc_hoa_trang',
        'desc': 'Nghệ nhân mặc quần áo vải nâu mộc mạc, mũ cát, đeo kính đen (Xẩm cổ / Xẩm tàu điện)',
        'default_genres': ['Hát xẩm']
    },

    # Nhóm 3: Không gian & Đội hình (6 attributes)
    'san_khau_kich_hat': {
        'group': 'khong_gian_doi_hinh',
        'desc': 'Biểu diễn trên sân khấu kịch hát có phông màn, cánh gà, ánh sáng phục vụ cốt truyện',
        'default_genres': ['Cải lương']
    },
    'doi_hinh_3_nguoi_ca_tru': {
        'group': 'khong_gian_doi_hinh',
        'desc': 'Đội hình thính phòng chuẩn 3 người: đào nương, kép đàn đáy và quan viên trống chầu',
        'default_genres': ['Ca trù']
    },
    'chieu_cheo_san_dinh': {
        'group': 'khong_gian_doi_hinh',
        'desc': 'Chiếu Chèo trải ở sân đình, sân làng hoặc nhà hát kịch hát dân gian',
        'default_genres': ['Chèo']
    },
    'dien_tho_hau_dong': {
        'group': 'khong_gian_doi_hinh',
        'desc': 'Không gian nghi lễ tâm linh thờ Mẫu tại đền, phủ, điện thờ',
        'default_genres': ['Chầu văn']
    },
    'doi_dap_nam_nu_quan_ho': {
        'group': 'khong_gian_doi_hinh',
        'desc': 'Hát đối đáp giao duyên theo cặp/nhóm nam nữ (liền anh - liền chị) đối diện nhau',
        'default_genres': ['Quan họ']
    },
    'chieu_xam_duong_pho_cho': {
        'group': 'khong_gian_doi_hinh',
        'desc': 'Chiếu Xẩm bình dị trải tại chợ, đường phố, bến đò hoặc toa tàu điện đô thị',
        'default_genres': ['Hát xẩm']
    }
}

CONCEPT_COLS = list(VISUAL_CONCEPT_ATTRIBUTES.keys())
print(f'Đã khởi tạo thành công {len(CONCEPT_COLS)} thuộc tính Visual Concepts cốt lõi!')


### BƯỚC 2: Khai Thác & Chuyển Đổi Tập `roboflowdataset` (`_annotations.coco.json`)
- Đọc file `_annotations.coco.json` trong 3 tập `train`, `valid`, `test`.
- Lọc 21 categories hữu ích trong 107 classes và ánh xạ sang các cột Visual Concepts nhị phân (0/1):
  - `dan bau` (96) -> `dan_bau = 1`
  - `dan co` (97) -> `dan_co_dan_nhi = 1`
  - `dan guitar` (98) -> `guitar_phim_lom = 1`
  - `dan kim` (99) -> `dan_kim_dan_nguyet = 1`
  - `dan tranh` (102) -> `dan_tranh = 1`
  - `Trong` (87) -> `trong_cheo_mo_quat_giay = 1`
  - `Ao dai` (2), `Ao dai truyen thong` (3) -> `ao_dai_ao_ba_ba = 1`
  - `Dau doi khan dong` (31) -> `ao_thao_khan_nhieu_ao_the = 1`
  - `Dau doi vuong mien` (32), `Giap nguc dinh hat` (43), `Mac do dinh hat` (56), `Tay ao dinh hat` (77), `Nghe si Hat Boi` (61) -> `co_trang_mao_giap_dinh_hat = 1`
  - `Cam vu khi` (11), `Cam gay` (10), `Mam cung` (57) -> `dao_cu_hau_dong_kiem_cung_mam = 1`
  - `San khau Hat Boi` (75) -> `san_khau_kich_hat = 1`
- Đồng thời trích xuất các bounding box sang định dạng **YOLOv8** (`data/yolo_dataset/`) kèm file `data.yaml` để sẵn sàng huấn luyện mô hình phát hiện đạo cụ tự động `YOLOv8n-Heritage-Props`!

In [ ]:
# 2. KHAI THÁC ROBOFLOW DATASET & CHUYỂN ĐỔI SANG VISUAL CONCEPTS VÀ YOLO BBOX
# Ánh xạ tên category Roboflow sang Concept Attribute tương ứng
ROBOFLOW_CATEGORY_TO_CONCEPT = {
    'dan bau': 'dan_bau',
    'dan co': 'dan_co_dan_nhi',
    'dan guitar': 'guitar_phim_lom',
    'dan kim': 'dan_kim_dan_nguyet',
    'dan tranh': 'dan_tranh',
    'Trong': 'trong_cheo_mo_quat_giay',
    'Ao dai': 'ao_dai_ao_ba_ba',
    'Ao dai truyen thong': 'ao_dai_ao_ba_ba',
    'Dau doi khan dong': 'ao_thao_khan_nhieu_ao_the',
    'Dau doi vuong mien': 'co_trang_mao_giap_dinh_hat',
    'Giap nguc dinh hat': 'co_trang_mao_giap_dinh_hat',
    'Mac do dinh hat': 'co_trang_mao_giap_dinh_hat',
    'Tay ao dinh hat': 'co_trang_mao_giap_dinh_hat',
    'Nghe si Hat Boi': 'co_trang_mao_giap_dinh_hat',
    'Cam vu khi': 'dao_cu_hau_dong_kiem_cung_mam',
    'Cam gay': 'dao_cu_hau_dong_kiem_cung_mam',
    'Mam cung': 'dao_cu_hau_dong_kiem_cung_mam',
    'San khau Hat Boi': 'san_khau_kich_hat'
}

# Danh sách lớp rút gọn phục vụ huấn luyện YOLOv8 Props Detector
YOLO_CLASSES = [
    'dan_bau',
    'dan_co_nhi',
    'dan_guitar',
    'dan_kim_nguyet',
    'dan_tranh',
    'trong',
    'ao_dai',
    'khan_dong',
    'co_trang_dinh_hat',
    'dao_cu_nghi_le',
    'san_khau_hat_boi'
]

YOLO_CAT_MAP = {
    'dan bau': 0,
    'dan co': 1,
    'dan guitar': 2,
    'dan kim': 3,
    'dan tranh': 4,
    'Trong': 5,
    'Ao dai': 6,
    'Ao dai truyen thong': 6,
    'Dau doi khan dong': 7,
    'Dau doi vuong mien': 8,
    'Giap nguc dinh hat': 8,
    'Mac do dinh hat': 8,
    'Tay ao dinh hat': 8,
    'Nghe si Hat Boi': 8,
    'Cam vu khi': 9,
    'Cam gay': 9,
    'Mam cung': 9,
    'San khau Hat Boi': 10
}

roboflow_records = []
total_converted_yolo = 0

print("Bắt đầu xử lý tập Roboflow Dataset...")
for split in ['train', 'valid', 'test']:
    split_dir = ROBOFLOW_DIR / split
    anno_file = split_dir / "_annotations.coco.json"
    if not anno_file.exists():
        print(f"  [Bỏ qua] Không tìm thấy: {anno_file}")
        continue

    with open(anno_file, 'r', encoding='utf-8') as f:
        coco_data = json.load(f)

    cat_id_to_name = {c['id']: c['name'] for c in coco_data['categories']}
    img_id_to_info = {img['id']: img for img in coco_data['images']}

    # Nhóm annotations theo image_id
    img_to_annos = {}
    for a in coco_data['annotations']:
        img_to_annos.setdefault(a['image_id'], []).append(a)

    # Thư mục đích YOLO
    yolo_split_img_dir = YOLO_DIR / split / "images"
    yolo_split_lbl_dir = YOLO_DIR / split / "labels"
    yolo_split_img_dir.mkdir(parents=True, exist_ok=True)
    yolo_split_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_id, img_info in img_to_annos.items():
        fname = img_id_to_info[img_id]['file_name']
        src_img_path = split_dir / fname
        if not src_img_path.exists():
            continue

        img_w = float(img_id_to_info[img_id]['width'])
        img_h = float(img_id_to_info[img_id]['height'])

        # Khởi tạo vector nhãn 25 concepts bằng 0
        img_concepts = {col: 0 for col in CONCEPT_COLS}
        yolo_lines = []
        has_useful_prop = False

        for ann in img_to_annos[img_id]:
            cname = cat_id_to_name.get(ann['category_id'], '')
            if cname in ROBOFLOW_CATEGORY_TO_CONCEPT:
                concept_attr = ROBOFLOW_CATEGORY_TO_CONCEPT[cname]
                img_concepts[concept_attr] = 1
                has_useful_prop = True

            # Chuyển đổi bounding box sang YOLO: [class_id, x_center, y_center, w, h]
            if cname in YOLO_CAT_MAP:
                bbox = ann['bbox'] # [x_min, y_min, w, h]
                x_min, y_min, bw, bh = bbox
                x_c = (x_min + bw / 2.0) / img_w
                y_c = (y_min + bh / 2.0) / img_h
                norm_w = bw / img_w
                norm_h = bh / img_h
                cls_id = YOLO_CAT_MAP[cname]
                yolo_lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {norm_w:.6f} {norm_h:.6f}")

        if has_useful_prop:
            record = {
                'image_path': str(src_img_path.relative_to(ROOT_DIR) if ROOT_DIR != Path('.') else src_img_path),
                'split': split,
                'file_name': fname,
                'width': int(img_w),
                'height': int(img_h)
            }
            record.update(img_concepts)
            roboflow_records.append(record)

            # Sao chép và xuất file label YOLO nếu có bbox
            if yolo_lines:
                lbl_fname = Path(fname).stem + ".txt"
                with open(yolo_split_lbl_dir / lbl_fname, 'w', encoding='utf-8') as lf:
                    lf.write("\n".join(yolo_lines))
                # Tạo symlink hoặc sao chép ảnh sang YOLO dataset
                dst_img = yolo_split_img_dir / fname
                if not dst_img.exists():
                    shutil.copy2(src_img_path, dst_img)
                total_converted_yolo += 1

    print(f"  -> Xong split '{split}': {len(img_to_annos)} ảnh duyệt, {len([r for r in roboflow_records if r['split']==split])} ảnh có concept hợp lệ.")

df_roboflow_concepts = pd.DataFrame(roboflow_records)
csv_roboflow_out = ANNOTATIONS_DIR / "roboflow_image_concepts.csv"
df_roboflow_concepts.to_csv(csv_roboflow_out, index=False)

# Tạo file data.yaml cho YOLOv8 training
yolo_yaml_path = YOLO_DIR / "data.yaml"
with open(yolo_yaml_path, 'w', encoding='utf-8') as yf:
    yf.write(f"path: {YOLO_DIR.resolve()}\n")
    yf.write("train: train/images\n")
    yf.write("val: valid/images\n")
    yf.write("test: test/images\n\n")
    yf.write(f"nc: {len(YOLO_CLASSES)}\n")
    yf.write(f"names: {YOLO_CLASSES}\n")

print(f"\n=> Đã lưu: {csv_roboflow_out} ({len(df_roboflow_concepts)} ảnh)")
print(f"=> Đã tạo tập YOLO Dataset: {total_converted_yolo} ảnh kèm nhãn tại: {YOLO_DIR}")
print(f"=> File cấu hình huấn luyện YOLO: {yolo_yaml_path}")

### BƯỚC 3: Trích Xuất Frame Thích Ứng từ `dataset/video/` (Xử Lý Video Lệch Thời Lượng & Chống Data Leakage)
1. **Incremental Ingestion Thông Minh:**
   - Quản lý sổ đăng ký `data/annotations/processed_videos_registry.json`.
   - Khi thêm video mới: Hệ thống tự động nhận diện và **BỎ QUA** toàn bộ video cũ (không trích xuất lại), chỉ xử lý video mới và tự động gán vào tập `train`.
2. **Cơ chế Virtual Sub-clips chống Data Leakage:**
   - Video <= 90 giây: giữ nguyên là 1 sub-clip.
   - Video dài 6-20 phút: chia thành các **Virtual Sub-clips dài 60 giây**, giữa 2 sub-clip liên tiếp có **khoảng đệm (buffer gap) 15 giây bị loại bỏ hoàn toàn**.
3. **Phân chia Tập Dữ Liệu KẾT HỢP TÊN TỆP VIDEO VÀ THỜI LƯỢNG (Duration-Aware Partitioning):**
   - Phân chia ở cấp độ **TÊN TỆP VIDEO GỐC (`video_file`)**, đảm bảo 100% Zero Data Leakage.
   - Cân bằng theo thời lượng: Test ~18%, Val ~14%, Train ~68%.
4. **Lọc Trùng Lặp Cảnh & Frame Mờ (Deduplication & Blur Filter):**
   - Lấy 1 frame mỗi 3 giây, lọc frame mờ (`Laplacian variance < 60`), lọc frame trùng lặp (`Color Histogram Sim > 0.92`), giới hạn tối đa 15 frames/subclip.

In [ ]:
# 3. TRÍCH XUẤT FRAME THÍCH ỨNG VÀ CHỐNG DATA LEAKAGE TỪ DATASET/VIDEO/
REGISTRY_PATH = ANNOTATIONS_DIR / 'processed_videos_registry.json'

# Nạp sổ đăng ký nếu đã tồn tại
if REGISTRY_PATH.exists():
    with open(REGISTRY_PATH, 'r', encoding='utf-8') as rf:
        processed_registry = json.load(rf)
    print(f"[Incremental Ingestion] Đã nạp sổ đăng ký: {len(processed_registry)} video đã xử lý trước đó.")
else:
    processed_registry = {}
    print("[Initial Ingestion] Chưa có sổ đăng ký, khởi tạo mới...")

def get_video_subclips_and_duration(video_path: Path, min_clip_len=45, clip_len=60, gap_len=15):
    """
    Tính thời lượng video và chia thành các Virtual Sub-clips kèm Buffer Gap chống rò rỉ bối cảnh.
    """
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration = frame_count / fps if fps > 0 else 0.0
    cap.release()

    if duration <= 0:
        return [], 0.0

    if duration <= 90:
        return [{
            'subclip_idx': 0,
            'start_sec': 0.0,
            'end_sec': duration,
            'duration': duration
        }], duration

    subclips = []
    curr_start = 0.0
    idx = 0
    while curr_start + min_clip_len <= duration:
        curr_end = min(curr_start + clip_len, duration)
        subclips.append({
            'subclip_idx': idx,
            'start_sec': curr_start,
            'end_sec': curr_end,
            'duration': curr_end - curr_start
        })
        curr_start += (clip_len + gap_len)
        idx += 1

    return subclips, duration

def is_frame_sharp_and_bright(frame_bgr, min_var=60.0):
    """Kiểm tra frame có đủ độ nét và không bị màn hình đen/trắng chuyển cảnh."""
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    var = cv2.Laplacian(gray, cv2.CV_64F).var()
    mean_val = gray.mean()
    if var < min_var or mean_val < 15.0 or mean_val > 240.0:
        return False, var
    return True, var

def compute_color_hist(frame_bgr):
    """Tính Color Histogram chuẩn hóa (HSV) dùng so sánh độ tương đồng cảnh."""
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [16, 16], [0, 180, 0, 256])
    cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    return hist

def duration_aware_video_split(genre_video_items, target_test_ratio=0.18, target_val_ratio=0.14, seed=42):
    """
    Phân chia tập dữ liệu cấp độ Video kết hợp Ngân sách Thời lượng (Duration-Aware Video Partitioning).
    """
    rng = random.Random(seed)
    total_sec = sum(d for _, d in genre_video_items)
    target_test_sec = total_sec * target_test_ratio
    target_val_sec = total_sec * target_val_ratio

    available = genre_video_items.copy()
    min_test_vids = 2 if len(available) >= 4 else 1
    best_test_subset = None
    best_test_diff = float('inf')

    for _ in range(500):
        sample_pool = available.copy()
        rng.shuffle(sample_pool)
        curr_subset = []
        curr_d = 0.0
        for vf, d in sample_pool:
            if curr_d + d <= target_test_sec * 1.25 or len(curr_subset) < min_test_vids:
                curr_subset.append((vf, d))
                curr_d += d
            if curr_d >= target_test_sec * 0.85 and len(curr_subset) >= min_test_vids:
                break
        diff = abs(curr_d - target_test_sec)
        if diff < best_test_diff and len(curr_subset) >= min_test_vids:
            best_test_diff = diff
            best_test_subset = curr_subset

    test_names = {vf for vf, _ in best_test_subset}
    remain_pool = [x for x in available if x[0] not in test_names]

    min_val_vids = 1 if len(remain_pool) >= 3 else 0
    best_val_subset = []
    best_val_diff = float('inf')

    if min_val_vids > 0:
        for _ in range(500):
            sample_pool = remain_pool.copy()
            rng.shuffle(sample_pool)
            curr_subset = []
            curr_d = 0.0
            for vf, d in sample_pool:
                if curr_d + d <= target_val_sec * 1.25 or len(curr_subset) < min_val_vids:
                    curr_subset.append((vf, d))
                    curr_d += d
                if curr_d >= target_val_sec * 0.85 and len(curr_subset) >= min_val_vids:
                    break
            diff = abs(curr_d - target_val_sec)
            if diff < best_val_diff and len(curr_subset) >= min_val_vids:
                best_val_diff = diff
                best_val_subset = curr_subset

    val_names = {vf for vf, _ in best_val_subset}
    train_subset = [x for x in remain_pool if x[0] not in val_names]

    return train_subset, best_val_subset, best_test_subset, total_sec

if not VIDEO_DATASET_DIR.exists():
    raise FileNotFoundError(f'Không tìm thấy thư mục video tại: {VIDEO_DATASET_DIR.resolve()}')

all_subclips_metadata = []
genre_video_durations = {}
print(f'Bắt đầu quét danh mục video từ: {VIDEO_DATASET_DIR.resolve()}...')

for genre_dir_name in sorted(os.listdir(VIDEO_DATASET_DIR)):
    genre_dir = VIDEO_DATASET_DIR / genre_dir_name
    if not genre_dir.is_dir():
        continue

    genre_clean = genre_dir_name.lower().replace(' ', '').replace('-', '')
    genre_name = GENRE_MAP.get(genre_clean, GENRE_MAP.get(genre_dir_name, None))
    if genre_name is None:
        for k, v in GENRE_MAP.items():
            if k in genre_clean:
                genre_name = v
                break
    if genre_name is None:
        continue

    v_files = [f for f in os.listdir(genre_dir) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv', '.webm'))]
    if not v_files:
        continue

    genre_video_durations[genre_name] = []
    for vf in v_files:
        v_path = genre_dir / vf
        clips, dur_sec = get_video_subclips_and_duration(v_path)
        genre_video_durations[genre_name].append((vf, dur_sec))

        for c in clips:
            subclip_id = f"{v_path.stem}_sub{c['subclip_idx']:02d}"
            all_subclips_metadata.append({
                'subclip_id': subclip_id,
                'genre': genre_name,
                'genre_code': genre_dir_name,
                'video_file': vf,
                'video_path': str(v_path),
                'start_sec': c['start_sec'],
                'end_sec': c['end_sec'],
                'duration': c['duration']
            })

if len(all_subclips_metadata) == 0:
    print('[CẢNH BÁO] Không tìm thấy video nào để trích xuất frames!')
    df_all_subclips = pd.DataFrame()
    df_extracted_frames = pd.DataFrame()
else:
    df_all_subclips = pd.DataFrame(all_subclips_metadata)

    # Phân chia hoặc kế thừa Split cho từng video
    video_split_map = {}
    is_initial_run = (len(processed_registry) == 0)

    if is_initial_run:
        print('\n' + '='*85)
        print('BÁO CÁO PHÂN CHIA DỮ LIỆU KẾT HỢP TÊN TỆP VIDEO & NGÂN SÁCH THỜI LƯỢNG (DURATION-AWARE)')
        print('='*85)
        for g, vid_items in genre_video_durations.items():
            tr, va, te, tot_sec = duration_aware_video_split(vid_items, target_test_ratio=0.18, target_val_ratio=0.14, seed=SEED)
            for v, _ in tr: video_split_map[v] = 'train'
            for v, _ in va: video_split_map[v] = 'val'
            for v, _ in te: video_split_map[v] = 'test'

            tot_m = tot_sec / 60.0
            tr_m = sum(x[1] for x in tr) / 60.0
            va_m = sum(x[1] for x in va) / 60.0
            te_m = sum(x[1] for x in te) / 60.0
            print(f'* Thể loại: [{g}] | Train: {tr_m:.1f}m ({len(tr)} vids) | Val: {va_m:.1f}m ({len(va)} vids) | Test: {te_m:.1f}m ({len(te)} vids)')
    else:
        print('\n[Incremental Ingestion] Bảo toàn Split cho video cũ, tự động gán video mới vào Train...')
        new_cnt = 0
        for g, vid_items in genre_video_durations.items():
            for vf, _ in vid_items:
                if vf in processed_registry:
                    video_split_map[vf] = processed_registry[vf].get('split', 'train')
                else:
                    video_split_map[vf] = 'train'
                    new_cnt += 1
        print(f'  -> Số video mới được phát hiện: {new_cnt} video (được thêm vào tập Train).')

    df_all_subclips['split'] = df_all_subclips['video_file'].map(video_split_map)

    # Trích xuất Frame (Bỏ qua các video đã có trong registry)
    extracted_frames_records = []
    max_frames_per_subclip = 15
    sample_rate_sec = 3.0
    reused_subclips = 0
    extracted_subclips = 0

    print('\nBắt đầu trích xuất Frame thích ứng (Deduplication + Blur Filtering)...')
    for idx, row in tqdm(df_all_subclips.iterrows(), total=len(df_all_subclips)):
        subclip_id = row['subclip_id']
        genre_code = row['genre_code']
        v_path = Path(row['video_path'])
        v_file = row['video_file']
        split = row['split']

        out_genre_dir = FRAMES_DIR / genre_code
        out_genre_dir.mkdir(parents=True, exist_ok=True)

        # Kiểm tra nếu video cũ đã xử lý và còn đầy đủ frames
        if not is_initial_run and v_file in processed_registry:
            reg_subclips = processed_registry[v_file].get('subclips', {})
            if subclip_id in reg_subclips:
                meta = reg_subclips[subclip_id]
                df_all_subclips.loc[idx, 'thumbnail_path'] = meta.get('thumbnail_path', '')
                df_all_subclips.loc[idx, 'frame_count'] = meta.get('frame_count', 0)
                for fr in meta.get('frames', []):
                    extracted_frames_records.append(fr)
                reused_subclips += 1
                continue

        cap = cv2.VideoCapture(str(v_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        start_f = int(row['start_sec'] * fps)
        end_f = int(row['end_sec'] * fps)
        step_f = max(1, int(sample_rate_sec * fps))

        saved_frames_in_subclip = 0
        prev_hist = None
        subclip_thumbnail_path = None
        max_sharpness = -1.0
        current_sub_frames = []

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
        cur_f = start_f

        while cur_f < end_f and saved_frames_in_subclip < max_frames_per_subclip:
            ret, frame = cap.read()
            if not ret or frame is None:
                break

            is_sharp, var_val = is_frame_sharp_and_bright(frame, min_var=60.0)
            if is_sharp:
                curr_hist = compute_color_hist(frame)
                is_duplicate = False
                if prev_hist is not None:
                    sim = cv2.compareHist(prev_hist, curr_hist, cv2.HISTCMP_CORREL)
                    if sim > 0.92:
                        is_duplicate = True

                if not is_duplicate:
                    prev_hist = curr_hist
                    frame_filename = f"{subclip_id}_f{saved_frames_in_subclip:02d}.jpg"
                    save_path = out_genre_dir / frame_filename
                    cv2.imwrite(str(save_path), frame)

                    rel_img_path = str(save_path.relative_to(ROOT_DIR) if ROOT_DIR != Path('.') else save_path)
                    f_dict = {
                        'image_path': rel_img_path,
                        'video_file': v_file,
                        'subclip_id': subclip_id,
                        'genre': row['genre'],
                        'genre_code': genre_code,
                        'split': split,
                        'frame_idx': saved_frames_in_subclip,
                        'sharpness': round(var_val, 2)
                    }
                    extracted_frames_records.append(f_dict)
                    current_sub_frames.append(f_dict)

                    if var_val > max_sharpness:
                        max_sharpness = var_val
                        subclip_thumbnail_path = rel_img_path

                    saved_frames_in_subclip += 1

            cur_f += step_f
            cap.set(cv2.CAP_PROP_POS_FRAMES, cur_f)

        cap.release()
        thumb = subclip_thumbnail_path or ''
        df_all_subclips.loc[idx, 'thumbnail_path'] = thumb
        df_all_subclips.loc[idx, 'frame_count'] = saved_frames_in_subclip
        extracted_subclips += 1

        # Lưu lại thông tin subclip vào registry
        if v_file not in processed_registry:
            processed_registry[v_file] = {'genre': row['genre'], 'split': split, 'subclips': {}}
        processed_registry[v_file]['subclips'][subclip_id] = {
            'thumbnail_path': thumb,
            'frame_count': saved_frames_in_subclip,
            'frames': current_sub_frames
        }

    # Ghi lại sổ đăng ký
    with open(REGISTRY_PATH, 'w', encoding='utf-8') as wf:
        json.dump(processed_registry, wf, indent=2, ensure_ascii=False)
    print(f'\n=> Sổ đăng ký đã cập nhật: {REGISTRY_PATH} ({reused_subclips} subclips tái sử dụng, {extracted_subclips} subclips mới).')

    df_extracted_frames = pd.DataFrame(extracted_frames_records)
    print(f'=> Sẵn sàng {len(df_extracted_frames)} frames đại diện (đã lọc mờ & trùng lặp)!')
    if not df_extracted_frames.empty:
        print(pd.crosstab(df_extracted_frames['genre'], df_extracted_frames['split']))


### BƯỚC 4.1: TẦNG 1 – Xuất & Bảo Toàn Sub-clip Level Review Sheet (Hướng 2 Bán Tự Động)
Cell này tạo hoặc nạp lại file `data/annotations/subclip_review_sheet.csv`:
- **Bảo toàn chỉnh sửa:** Nếu file đã tồn tại (bạn đã từng chỉnh sửa tay), code sẽ **BẢO TOÀN NGUYÊN VẸN** các chỉnh sửa cũ và chỉ bổ sung thêm các phân đoạn của video mới.
- **Xem trước hình ảnh:** Cột `thumbnail_path` cho phép bạn mở ảnh nét nhất của từng phân đoạn 60s để kiểm tra bối cảnh, đạo cụ, trang phục.
- **Sau khi chạy Cell này:** Bạn hãy mở file `subclip_review_sheet.csv` để tự điều chỉnh số 0/1 trước khi chạy tiếp Bước 4.2!

In [ ]:
# 4.1. TẦNG 1: XUẤT SUB-CLIP LEVEL REVIEW SHEET (HƯỚNG 2)
print('4.1. Đang chuẩn bị Sub-clip Level Review Sheet (Tầng 1)...')

subclip_sheet_path = ANNOTATIONS_DIR / 'subclip_review_sheet.csv'

# Điền sẵn giá trị mặc định cho từng sub-clip dựa trên Genre Prior
for col in CONCEPT_COLS:
    default_genres = VISUAL_CONCEPT_ATTRIBUTES[col]['default_genres']
    df_all_subclips[col] = df_all_subclips['genre'].apply(lambda g: 1 if g in default_genres else 0)

review_cols = ['subclip_id', 'genre', 'video_file', 'split', 'duration', 'frame_count', 'thumbnail_path'] + CONCEPT_COLS
df_current_subclips = df_all_subclips[[c for c in review_cols if c in df_all_subclips.columns]].copy()

if subclip_sheet_path.exists():
    print(f'  [Thông báo] Đã tìm thấy file subclip_review_sheet.csv đã có!')
    try:
        df_existing_sheet = pd.read_csv(subclip_sheet_path)
        existing_ids = set(df_existing_sheet['subclip_id'])
        new_subclips = df_current_subclips[~df_current_subclips['subclip_id'].isin(existing_ids)]

        if not new_subclips.empty:
            print(f'  -> Phát hiện {len(new_subclips)} sub-clips mới từ video vừa bổ sung. Đang ghép thêm vào sheet...')
            df_subclip_sheet = pd.concat([df_existing_sheet, new_subclips], ignore_index=True)
            df_subclip_sheet.to_csv(subclip_sheet_path, index=False, encoding='utf-8-sig')
            print(f'  -> Đã cập nhật {subclip_sheet_path} (BẢO TOÀN {len(df_existing_sheet)} sub-clips cũ đã chỉnh sửa tay!)')
        else:
            df_subclip_sheet = df_existing_sheet
            print(f'  -> Bảo toàn nguyên vẹn {len(df_subclip_sheet)} sub-clips hiện tại trong file CSV.')
    except Exception as e:
        print(f'  [Cảnh báo] Lỗi khi đọc sheet cũ ({e}). Đang tạo lại mới...')
        df_subclip_sheet = df_current_subclips
        df_subclip_sheet.to_csv(subclip_sheet_path, index=False, encoding='utf-8-sig')
else:
    df_subclip_sheet = df_current_subclips
    df_subclip_sheet.to_csv(subclip_sheet_path, index=False, encoding='utf-8-sig')
    print(f'  -> Đã tạo mới Sub-clip Review Sheet: {subclip_sheet_path} ({len(df_subclip_sheet)} sub-clips)')

print('\n' + '='*85)
print('HƯỚNG DẪN KIỂM DUYỆT & TỰ ĐIỀU CHỈNH PHÂN ĐOẠN:')
print('='*85)
print(f'1. Mở file CSV tại: {subclip_sheet_path.resolve()}')
print('   (Bạn có thể mở trực tiếp trong Google Colab, Google Drive, Excel hoặc VS Code).')
print('2. Trong file này:')
print('   - Cột `thumbnail_path`: Ảnh đại diện nét nhất của phân đoạn 60s đó.')
print('   - 25 cột concept (guitar_phim_lom, dan_tranh, co_trang_mao_giap_dinh_hat, ...): Đang có giá trị 0 hoặc 1.')
print('3. Bạn chỉ cần sửa số 0 hoặc 1:')
print('   - Phân đoạn thực tế KHÔNG có đàn kìm -> sửa cột `dan_kim_dan_nguyet` thành 0.')
print('   - Phân đoạn CÓ xuất hiện múa kiếm/bắn cung -> sửa cột `dao_cu_hau_dong_kiem_cung_mam` thành 1.')
print('4. Nhấn Ctrl + S để LƯU FILE CSV.')
print('5. SAU ĐÓ BẤM CHẠY CELL BƯỚC 4.2 (TẦNG 2) BÊN DƯỚI ĐỂ ÁP DỤNG!')
print('='*85)


### BƯỚC 4.2: TẦNG 2 – Nạp Sheet Đã Chỉnh Sửa, Tinh Chỉnh CLIP & Hợp Nhất Dataset Cuối
> **LƯU Ý:** Trước khi chạy cell bên dưới, hãy đảm bảo bạn đã lưu file `data/annotations/subclip_review_sheet.csv` nếu có chỉnh sửa tay.
> Cell này sẽ **đọc trực tiếp từ file CSV trên đĩa**, chạy tinh chỉnh cấp Frame bằng Zero-shot CLIP, ghép ảnh Roboflow và xuất ra `visual_concepts_dataset.csv`.

In [ ]:
# 4.2. TẦNG 2: FRAME-LEVEL REFINEMENT BẰNG ZERO-SHOT CLIP & XUẤT DATASET CUỐI
print('4.2. Đang nạp Sub-clip Review Sheet trực tiếp từ đĩa để nhận các chỉnh sửa tay...')

subclip_sheet_path = ANNOTATIONS_DIR / 'subclip_review_sheet.csv'
if not subclip_sheet_path.exists():
    raise FileNotFoundError(f'Chưa có file {subclip_sheet_path}. Vui lòng chạy Bước 4.1 trước!')

# ĐỌC TRỰC TIẾP TỪ ĐĨA ĐỂ LẤY TOÀN BỘ CHỈNH SỬA TAY CỦA BẠN
df_subclip_sheet = pd.read_csv(subclip_sheet_path)
print(f'  -> Đã nạp thành công {len(df_subclip_sheet)} sub-clips từ: {subclip_sheet_path.name}')

print('\nKhởi tạo mô hình Zero-shot CLIP để tinh chỉnh nhãn frame riêng lẻ...')
has_clip = False
try:
    from transformers import CLIPProcessor, CLIPModel
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
    clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    has_clip = True
    print('  -> Đã nạp thành công mô hình Zero-shot CLIP (openai/clip-vit-base-patch32)!')
except Exception as e:
    print(f'  [Thông báo] Không thể tải CLIP ({e}). Sử dụng nhãn kế thừa từ Sub-clip Level.')

# Bảng mô tả tiếng Anh chuẩn để CLIP zero-shot matching cho 25 concepts
CLIP_PROMPTS = {
    'guitar_phim_lom': 'Vietnamese scalloped fretboard guitar in Cai Luong opera performance',
    'dan_kim_dan_nguyet': 'Vietnamese moon lute dan nguyet or dan kim musical instrument',
    'dan_co_dan_nhi': 'Vietnamese two string vertical fiddle dan co or dan nhi',
    'dan_tranh': 'Vietnamese 16 string zither dan tranh musical instrument',
    'dan_bau': 'Vietnamese monochord dan bau one string instrument',
    'dan_day': 'Vietnamese traditional long neck lute dan day in Ca tru',
    'ban_phach_trong_chau': 'wooden clapper phach and small praise drum trong chau in Ca tru',
    'trong_cheo_mo_quat_giay': 'Vietnamese folk opera Cheo drum, wooden slit drum and paper fan prop',
    'dao_cu_hau_dong_kiem_cung_mam': 'ritual sword, bow, gold trays and ceremonial props in Chau Van Mother Goddess ritual',
    'senh_tien_trong_manh': 'coin clapper senh tien and small blind busker drum in Hat Xam',
    'khong_nhac_cu': 'singers performing a cappella with no musical instruments in hands',
    'co_trang_mao_giap_dinh_hat': 'elaborate glittering imperial crown, beaded armor and silk costume in Vietnamese opera',
    'ao_dai_ao_ba_ba': 'traditional Vietnamese ao dai silk tunic or southern ao ba ba',
    'ao_thao_khan_nhieu_ao_the': 'traditional silk gown with head turban worn in Ca tru performance',
    'ao_tu_than_yem_dao': 'four flap dress ao tu than, pink halter bodice and skirt in northern folk opera',
    'trang_phuc_he_cheo': 'exaggerated colorful comic clown clown costume in folk theatre Cheo',
    'khan_chau_ao_ngu_hau_dong': 'vibrant multi colored sacred ritual robes, scarf and headdress in spirit possession',
    'non_quai_thao_khan_mo_qua_ao_mo_ba': 'large flat palm hat non quai thao, raven beak scarf and layered silk dresses in Quan ho',
    'ao_nau_kinh_den_mu_cat': 'rustic brown peasant clothes, black sunglasses and straw hat in blind busker Xam',
    'san_khau_kich_hat': 'theatrical drama stage with curtains, painted backdrop and stage lighting',
    'doi_hinh_3_nguoi_ca_tru': 'traditional chamber trio trio of female singer, lute player and praise drummer',
    'chieu_cheo_san_dinh': 'traditional communal village hall courtyard with mat for Cheo theatre',
    'dien_tho_hau_dong': 'sacred ornate altar temple and shrine sanctuary for Mother Goddess worship',
    'doi_dap_nam_nu_quan_ho': 'alternating courtship singing between male and female pairs facing each other',
    'chieu_xam_duong_pho_cho': 'rustic street singing mat on market pavement, ferry dock or urban tramway'
}

NEGATIVE_PROMPTS = [
    'a close up portrait of human face without any musical instruments or props',
    'a blurry background without any performers or cultural objects',
    'a blank stage empty of performers'
]

# Nạp bảng nhãn từ Sub-clip Review Sheet làm nhãn gốc
subclip_label_map = df_subclip_sheet.set_index('subclip_id').to_dict('index')

final_frame_dataset = []
print(f'\nBắt đầu tinh chỉnh nhãn trên {len(df_extracted_frames)} frames...')

for _, row in tqdm(df_extracted_frames.iterrows(), total=len(df_extracted_frames)):
    sub_id = row['subclip_id']
    img_rel_path = row['image_path']
    v_file = row.get('video_file', '')
    full_img_path = ROOT_DIR / img_rel_path

    # Lấy nhãn cơ sở từ Sub-clip Level (đã có chỉnh sửa tay của bạn)
    base_labels = subclip_label_map.get(sub_id, {})
    frame_concepts = {col: base_labels.get(col, 0) for col in CONCEPT_COLS}

    confidence_score = 0.85

    # Tinh chỉnh bằng CLIP nếu có ảnh và mô hình CLIP hoạt động
    if has_clip and full_img_path.exists():
        try:
            image = Image.open(full_img_path).convert('RGB')
            active_concepts = [c for c in CONCEPT_COLS if frame_concepts[c] == 1]
            if active_concepts:
                text_candidates = [CLIP_PROMPTS[c] for c in active_concepts] + NEGATIVE_PROMPTS
                inputs = clip_processor(text=text_candidates, images=image, return_tensors='pt', padding=True).to(device)
                with torch.no_grad():
                    outputs = clip_model(**inputs)
                    probs = outputs.logits_per_image.softmax(dim=1)[0].cpu().numpy()

                neg_prob = sum(probs[len(active_concepts):])
                if neg_prob > 0.65:
                    for c in active_concepts:
                        if VISUAL_CONCEPT_ATTRIBUTES[c]['group'] == 'nhac_cu_dao_cu' and c != 'khong_nhac_cu':
                            frame_concepts[c] = 0
                    confidence_score = 0.90
                else:
                    confidence_score = float(max(probs[:len(active_concepts)]))
        except Exception:
            pass

    record = {
        'image_path': img_rel_path,
        'video_file': v_file,
        'subclip_id': sub_id,
        'genre': row['genre'],
        'genre_code': row['genre_code'],
        'split': row['split'],
        'frame_idx': row['frame_idx'],
        'confidence': round(confidence_score, 3)
    }
    record.update(frame_concepts)
    final_frame_dataset.append(record)

df_visual_frames = pd.DataFrame(final_frame_dataset)

# -----------------------------------------------------------------------------
# 4.3. TỰ ĐỘNG HỢP NHẤT VIDEO FRAMES VÀ ROBOFLOW DATASET
# -----------------------------------------------------------------------------
rf_csv_path = ANNOTATIONS_DIR / 'roboflow_image_concepts.csv'
if rf_csv_path.exists():
    print(f'\nĐang hợp nhất video frames với tập Roboflow ({rf_csv_path.name})...')
    df_rf = pd.read_csv(rf_csv_path)
    def infer_rf_genre(r):
        if r.get('guitar_phim_lom', 0) == 1 or r.get('co_trang_mao_giap_dinh_hat', 0) == 1: return 'Cải lương'
        if r.get('dan_day', 0) == 1 or r.get('ban_phach_trong_chau', 0) == 1: return 'Ca trù'
        if r.get('dao_cu_hau_dong_kiem_cung_mam', 0) == 1: return 'Chầu văn'
        if r.get('trong_cheo_mo_quat_giay', 0) == 1 or r.get('trang_phuc_he_cheo', 0) == 1: return 'Chèo'
        if r.get('non_quai_thao_khan_mo_qua_ao_mo_ba', 0) == 1: return 'Quan họ'
        if r.get('senh_tien_trong_manh', 0) == 1 or r.get('ao_nau_kinh_den_mu_cat', 0) == 1: return 'Hát xẩm'
        return 'Cải lương'

    if 'genre' not in df_rf.columns:
        df_rf['genre'] = df_rf.apply(infer_rf_genre, axis=1)
    if 'video_file' not in df_rf.columns:
        df_rf['video_file'] = 'roboflow_static_image'
    if 'confidence' not in df_rf.columns:
        df_rf['confidence'] = 1.0

    if 'split' in df_rf.columns:
        df_rf['split'] = df_rf['split'].replace({'valid': 'val'})

    common_cols = [c for c in df_visual_frames.columns if c in df_rf.columns]
    df_visual_final = pd.concat([df_visual_frames, df_rf[common_cols]], ignore_index=True).fillna(0)
    print(f'  -> Đã hợp nhất: {len(df_visual_frames)} frames video + {len(df_rf)} ảnh Roboflow = {len(df_visual_final)} ảnh tổng.')
else:
    df_visual_final = df_visual_frames

final_dataset_path = ANNOTATIONS_DIR / 'visual_concepts_dataset.csv'
df_visual_final.to_csv(final_dataset_path, index=False, encoding='utf-8-sig')

print(f'\n=> HOÀN THÀNH TẬP DỮ LIỆU CUỐI CÙNG: {final_dataset_path}')
print(f'   Tổng số ảnh: {len(df_visual_final)} ảnh gán nhãn đa thuộc tính (Multi-label Visual Concepts).')
print(f'   Phân bổ Split:')
print(df_visual_final['split'].value_counts())
print('\n[LƯU Ý]: Bạn hãy mở tiếp notebook Visual_CBM_and_Rule_Extraction_Colab.ipynb để cập nhật mô hình CBM và Hệ Luật!')


### BƯỚC 5: Chẩn Đoán & Trực Quan Hóa Kết Quả Dữ Liệu
Vẽ biểu đồ phân bố 25 Visual Concepts trên 6 thể loại và hiển thị lưới ảnh mẫu kèm nhãn đã gán.

In [ ]:
# 5. CHẨN ĐOÁN & TRỰC QUAN HÓA KẾT QUẢ DỮ LIỆU
if not df_visual_final.empty:
    plt.style.use('default')
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 1. Phân bố số frame theo thể loại và split
    split_summary = pd.crosstab(df_visual_final['genre'], df_visual_final['split'])
    split_summary.plot(kind='bar', stacked=True, ax=axes[0], colormap='viridis', edgecolor='black')
    axes[0].set_title("Phân bố Số Frame theo Thể loại & Tập chia (Train/Val/Test)", fontsize=12, fontweight='bold')
    axes[0].set_ylabel("Số lượng frame ảnh")
    axes[0].set_xlabel("Thể loại di sản")
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].grid(axis='y', linestyle='--', alpha=0.6)

    # 2. Tần suất xuất hiện của Top 12 Visual Concepts
    concept_counts = df_visual_final[CONCEPT_COLS].sum().sort_values(ascending=False).head(12)
    concept_counts.plot(kind='barh', ax=axes[1], color='#2b5c8f', edgecolor='black')
    axes[1].set_title("Top 12 Visual Concepts Xuất Hiện Nhiều Nhất", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Số frame kích hoạt (nhãn = 1)")
    axes[1].invert_yaxis()
    axes[1].grid(axis='x', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

    # 3. Hiển thị Lưới Ảnh Mẫu (Grid 2x3) ngẫu nhiên
    sample_rows = df_visual_final.sample(min(6, len(df_visual_final)), random_state=42)
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()

    for idx, (_, r) in enumerate(sample_rows.iterrows()):
        img_p = ROOT_DIR / r['image_path']
        if img_p.exists():
            img = Image.open(img_p)
            axes[idx].imshow(img)
            active_c = [c for c in CONCEPT_COLS if r[c] == 1][:3]
            axes[idx].set_title(f"[{r['genre']}] ({r['split']})\n" + "\n".join(active_c), fontsize=9)
        axes[idx].axis('off')

    plt.suptitle("Mẫu Frame Ảnh & Nhãn Visual Concepts Sau Khi Xử Lý Thích Ứng", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("Notebook đã hoàn thành tất cả các bước chuẩn bị dữ liệu hình ảnh!")
else:
    print("Chưa có dữ liệu frame để trực quan hóa.")